# Lesson 1.4 — Helm CLI Basics

**Phase 1: Foundations**

---

## Learning Objectives

By the end of this lesson you will be able to:
- Search for charts using `helm search`
- Inspect chart contents before installing with `helm show`
- Override values at install time with `--set` and `-f`
- Understand the difference between `helm template` and `helm install --dry-run`
- Use `helm pull` to download a chart locally

---

## Searching for Charts

You can search both locally-added repositories and Artifact Hub (the central chart index):

In [ ]:
# Search Artifact Hub (online index of 800+ chart repos) for nginx charts
helm search hub nginx --max-col-width 80 2>/dev/null | head -20

In [ ]:
# Search only your locally added repositories
helm repo add bitnami https://charts.bitnami.com/bitnami 2>/dev/null || true
helm repo update
helm search repo nginx

---

## Inspecting a Chart Before Installing

The `helm show` command lets you peek inside a chart without installing it:

In [ ]:
# Show chart metadata (Chart.yaml)
helm show chart bitnami/nginx 2>/dev/null | head -30

In [ ]:
# Show the default values.yaml for the chart
# This is your reference for what can be customized
helm show values bitnami/nginx 2>/dev/null | head -60

In [ ]:
# Show the README
helm show readme bitnami/nginx 2>/dev/null | head -40

> **Tip:** Always run `helm show values <chart>` before installing. It tells you every configurable option and its default.

---

## Overriding Values

Two ways to supply custom values:

### 1. `--set key=value` — inline, single values

In [ ]:
# Install nginx and override replicaCount and service type
helm install demo-nginx bitnami/nginx \
  --namespace helm-study \
  --create-namespace \
  --set replicaCount=2 \
  --set service.type=ClusterIP \
  --dry-run 2>/dev/null | grep -A5 'replicas\|type:' | head -20

### 2. `-f values.yaml` — a file with multiple overrides

For non-trivial customisation, prefer a values file over chaining many `--set` flags:

In [ ]:
%%writefile /tmp/my-nginx-values.yaml
replicaCount: 3
service:
  type: ClusterIP
  port: 8080
resources:
  limits:
    cpu: "200m"
    memory: "128Mi"

In [ ]:
# Install using our custom values file
helm install demo-nginx bitnami/nginx \
  --namespace helm-study \
  --create-namespace \
  -f /tmp/my-nginx-values.yaml \
  --dry-run 2>/dev/null | grep -A3 'replicas\|port:\|cpu:' | head -30

### Value Override Precedence (highest to lowest)

```
--set            (wins)
  -f values.yaml
    chart's values.yaml defaults  (lowest priority)
```

You can layer multiple `-f` files; later files override earlier ones.

---

## `helm template` — Fully Offline Rendering

`helm template` renders charts to YAML **without contacting Kubernetes at all**. Perfect for CI pipelines and offline environments:

In [ ]:
# Render the chart to YAML locally (no cluster needed)
helm template my-app bitnami/nginx \
  --set replicaCount=2 \
  2>/dev/null | head -50

| | `helm template` | `helm install --dry-run` |
|---|---|---|
| Contacts cluster? | No | Yes (for validation) |
| Requires kubeconfig? | No | Yes |
| Use in CI? | Ideal | Requires cluster access |

---

## Downloading a Chart Locally

Use `helm pull` to download a chart archive (`.tgz`) to your machine:

In [ ]:
# Pull (download) the bitnami/nginx chart archive to /tmp
helm pull bitnami/nginx --destination /tmp
ls /tmp/nginx-*.tgz 2>/dev/null

In [ ]:
# Pull and untar in one step — useful for inspecting templates
helm pull bitnami/nginx --untar --destination /tmp/nginx-chart
ls /tmp/nginx-chart/nginx/

---

## Exercise

In [ ]:
# TASK 1: Use helm show values to find the name of the value that controls
#         the nginx container image repository in bitnami/nginx.
#         Hint: look for 'image:' in the output.
# TODO: write your command here


# TASK 2: Use helm template to render bitnami/nginx with replicaCount=3
#         and pipe the output through: grep 'replicas:'
# TODO: write your command here


# TASK 3: Create a values file that sets service.type to NodePort,
#         then render the chart with helm template using that file.
# TODO: write your commands here

---

## Summary

| Command | What It Does |
|---|---|
| `helm search hub <keyword>` | Search Artifact Hub |
| `helm search repo <keyword>` | Search local repos |
| `helm show chart <chart>` | Show Chart.yaml metadata |
| `helm show values <chart>` | Show configurable defaults |
| `helm show readme <chart>` | Show README |
| `helm install <name> <chart> --set k=v` | Install with inline value override |
| `helm install <name> <chart> -f vals.yaml` | Install with values file |
| `helm template <name> <chart>` | Render to YAML offline |
| `helm pull <chart>` | Download chart archive locally |

---

## Next Lesson

➡️ [Lesson 2.1 — Chart File Structure](../phase-2-chart-anatomy/lesson-2.1-chart-structure.ipynb)